# 03 Rare Events and Return Times

## Purpose

Apply threshold exceedances and recurrence diagnostics to a single empirical or fallback series.

## Data / config used

- local path: `data/raw/macro_stress_example.csv` if present
- synthetic fallback otherwise
- output directory: `outputs/notebooks/03_rare_events_and_return_times/`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from dynsys_econometrics.data import load_time_series_csv
from dynsys_econometrics.extremes import runs_extremal_index, threshold_exceedances
from dynsys_econometrics.return_times import return_times, survival_curve

output_dir = Path("outputs/notebooks/03_rare_events_and_return_times")
output_dir.mkdir(parents=True, exist_ok=True)

input_path = Path("data/raw/macro_stress_example.csv")
if input_path.exists():
    panel = load_time_series_csv(input_path, series_id="macro_stress")
else:
    dates = pd.date_range("2000-01-31", periods=220, freq="ME")
    values = np.cumsum(np.random.default_rng(21).normal(scale=0.2, size=len(dates))) + 0.4 * np.sin(np.linspace(0, 8, len(dates)))
    panel = pd.DataFrame({"date": dates, "series_id": "macro_stress", "value": values})


In [ ]:
events = threshold_exceedances(panel, quantile=0.95)
summary = runs_extremal_index(events, run_length=4)
durations = return_times(events)
curve = survival_curve(durations) if not durations.empty else pd.DataFrame(columns=["series_id", "duration", "survival_probability", "n_at_risk"])

events.to_csv(output_dir / "events.csv", index=False)
summary.to_csv(output_dir / "extremal_index.csv", index=False)
durations.to_csv(output_dir / "durations.csv", index=False)
curve.to_csv(output_dir / "survival_curve.csv", index=False)

summary


## Limitations

The diagnostics used here are descriptive and structural. They do not identify causal mechanisms by themselves, and short economic samples may make threshold-based recurrence summaries unstable.
